# Activity SOLUTION: Improving sentiment analysis with convolutional layers

> **Note**: This is the solution notebook. Compare your implementation with this one after completing the activity.

## Objective

In this activity, you will extend the baseline RNN sentiment classifier by adding **convolutional layers** to create a hybrid CNN-RNN architecture. Convolutional layers can capture local n-gram patterns (like "not good" or "very happy") before the recurrent layers process the sequence.

**Your tasks:**
1. Run the baseline GRU model and note its performance
2. Modify the model architecture to include convolutional layers
3. Train your CNN-RNN hybrid model
4. Compare the results to the baseline

---

## Key terms

- **Tokenization**: Splitting text into individual units (tokens), typically words. For example, "I love this!" becomes `["i", "love", "this"]`. This is the first step in converting text to a format neural networks can process.

- **Embedding**: A dense vector representation of a token. Instead of one-hot encoding (sparse, high-dimensional), embeddings map each word to a fixed-size vector (e.g., 100 dimensions) where similar words have similar vectors. We use pretrained GloVe embeddings trained on Twitter data.

- **LSTM/GRU**: Variants of RNNs designed to handle long sequences better than SimpleRNN. They use "gates" to control information flow, solving the vanishing gradient problem. **GRU** (Gated Recurrent Unit) has 2 gates and is simpler; **LSTM** (Long Short-Term Memory) has 3 gates and more parameters. Both work fine in practice, we use GRU here for efficiency.

## How the GRU processes a sequence

Given input sequence `["i", "love", "this]`, the GRU predicts sentiment:

```text

                   ┌────────────◀───────────┐ 'h₁', then 'h₂'
                   ▼                        │
                ╔══════════════════════════════╗
                ║  │           GRU          ▲  ║
                ║  │                        │  ║
  'I' then,     ║  ├──▶Reset───▶candidate──▶│  ║         ╔═══════╗ 
'love', then ══▶║  │   gate      hidden     │  ╠══'h₃'══▶║ Dense ╠══▶ Sentiment
   'this'       ║  │              state     │  ║         ╚═══════╝
                ║  │                        │  ║
                ║  └─────▶Update gate───────┘  ║
                ╚══════════════════════════════╝

```

- **Reset gate**: Controls how much the prior hidden state influences the new candidate hidden state. Gates are **dynamic weights**, one per hidden unit. Depends on **both** the previous hidden state AND the current input.
- **Update gate**: Controls how much prior hidden state vs new candidate hidden state goes into the new hidden state. Also computed from previous hidden state + current input, just with different learned weights.
- **Bidirectional recurrent layers**: Processes a sequence in both directions (forward and backward) and combines the results. This allows the model to use context from both before AND after each word, improving understanding.

## External tools & resources

- **Stopwords**: Common words (e.g., "the", "is", "at") filtered out during preprocessing to reduce noise and focus on meaningful content.
  - *This project*: [NLTK Stopwords](https://www.nltk.org/nltk_data/) curated lists for 20+ languages
  - *Alternatives*: [spaCy stopwords](https://spacy.io/usage/rule-based-matching#vocab-stopwords), [scikit-learn ENGLISH_STOP_WORDS](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.ENGLISH_STOP_WORDS.html)

- **Tokenizers**: Tools that split text into tokens. Different tokenizers handle edge cases differently social media text requires special handling for emoticons, hashtags, and informal spelling.
  - *This project*: [NLTK TweetTokenizer](https://www.nltk.org/api/nltk.tokenize.casual.html#nltk.tokenize.casual.TweetTokenizer) handles emoticons, hashtags, mentions, and normalizes repeated characters (e.g., "sooooo" → "sooo")
  - *Alternatives*: [spaCy Tokenizer](https://spacy.io/usage/linguistic-features#tokenization), [Hugging Face Tokenizers](https://huggingface.co/docs/tokenizers/), [NLTK word_tokenize](https://www.nltk.org/api/nltk.tokenize.word_tokenize.html)

- **Word Embeddings**: Pretrained vector representations that capture semantic relationships between words. Using pretrained embeddings transfers knowledge from large corpora to your model.
  - *This project*: [GloVe Twitter embeddings](https://nlp.stanford.edu/projects/glove/) trained on 27B Twitter tokens, 100 dimensions
  - *Alternatives*: [Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html), [FastText](https://fasttext.cc/docs/en/english-vectors.html), [Hugging Face sentence-transformers](https://huggingface.co/sentence-transformers)


## Notebook setup

### Imports

In [ ]:
import os
import re
import zipfile
import urllib.request
from collections import Counter
from pathlib import Path

import keras
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer
from keras.models import Sequential
from keras.layers import (
    Input, Embedding, Dense, Dropout, Bidirectional, GRU, SpatialDropout1D
)
from keras.preprocessing.sequence import pad_sequences
from keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.preprocessing import label_binarize

### GPU configuration

In [ ]:
# Configure GPU
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        # Set memory growth on ALL GPUs (required when multiple GPUs present)
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        print(f'Using {len(gpus)} GPU(s): {[gpu.name for gpu in gpus]}')
        
    except RuntimeError as e:
        print(e)

else:
    print('No GPU available, using CPU')

### Environment setup

In [ ]:
# Set data directory
data_dir = '../data'
Path(data_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
# Download NLTK resources
os.environ["NLTK_DATA"] = data_dir
nltk.download('stopwords', quiet=True)

In [ ]:
# GloVe Twitter embeddings download
glove_url  = 'https://nlp.stanford.edu/data/glove.twitter.27B.zip'
glove_file = os.path.join(data_dir, 'glove.twitter.27B.100d.txt')
glove_zip  = os.path.join(data_dir, 'glove.twitter.27B.zip')

if not os.path.exists(glove_file):

    print('Downloading GloVe embeddings...')
    urllib.request.urlretrieve(glove_url, glove_zip)

    print('Extracting...')
    with zipfile.ZipFile(glove_zip, 'r') as zip_ref:
        zip_ref.extract('glove.twitter.27B.100d.txt', data_dir)

    os.remove(glove_zip)
    print('Done!')

else:
    print(f'GloVe file already exists: {glove_file}')

### Shared hyperparameters

In [ ]:
hidden_dim              = 64      # Number of hidden units in the GRU layers
dropout                 = 0.3     # Spatial dropout rate
l2_reg                  = 0.02    # L2 regularization strength
learning_rate           = 0.0001  # Low LR for fine-tuning embeddings
batch_size              = 128
epochs                  = 200
early_stopping_patience = 20
verbose                 = 0

## 1. Data preparation

### 1.1. Load

In [ ]:
df = pd.read_parquet(os.path.join(data_dir, 'twitter-2016.parquet'))
df.head()

In [ ]:
df.info()

### 1.2. Clean

In [ ]:
# Clean data - drop rows with missing scores
df_clean = df.dropna(subset=['score']).copy()
df_clean['score'] = df_clean['score'].astype(int)

# Shift scores to 0-based index: [-2,-1,0,1,2] -> [0,1,2,3,4]
score_min = df_clean['score'].min()
df_clean['score_shifted'] = df_clean['score'] - score_min

print(f'Cleaned dataset: {len(df_clean):,} rows ({len(df) - len(df_clean):,} dropped)')
print(f'Score distribution: {df_clean["score_shifted"].value_counts().sort_index().to_dict()}')

### 1.3. Tokenize

In [ ]:
def clean_text(text):
    '''Clean and tokenize tweet text using TweetTokenizer.'''

    if not isinstance(text, str):
        return []

    # Remove URLs before tokenizing
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # TweetTokenizer handles: lowercase, @mentions, repeated chars (e.g., sooooo -> sooo)
    tokens = tweet_tokenizer.tokenize(text)

    # Remove '#' symbols from hashtags (keep the word)
    tokens = [t.lstrip('#') for t in tokens]

    # Remove stopwords and single characters
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]

    return tokens

In [ ]:
# Set up stop words and tokenizer
stop_words      = set(stopwords.words('english'))
tweet_tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)

# Apply tokenization
df_clean['tokens'] = df_clean['text'].apply(clean_text)
df_clean[['text', 'tokens', 'score']].head()

### 1.4. Build vocabulary

In [ ]:
# Build vocabulary
def build_vocab(token_lists, min_freq=2):
    '''Build vocabulary from token lists.'''

    counter = Counter()

    for tokens in token_lists:
        counter.update(tokens)
    
    # Filter by minimum frequency and create vocab
    vocab = {'<PAD>': 0, '<UNK>': 1}

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

vocab = build_vocab(df_clean['tokens'])
print(f'Vocabulary size: {len(vocab)}')

### 1.5. Convert to indices

In [ ]:
def tokens_to_indices(tokens_list, vocab, max_len=50):
    '''Convert token lists to padded index sequences.'''

    sequences = []

    for tokens in tokens_list:
        indices = [vocab.get(token, vocab['<UNK>']) for token in tokens]
        sequences.append(indices)

    # Pad sequences to max_len
    return pad_sequences(sequences, maxlen=max_len, padding='post', value=vocab['<PAD>'])

# Convert all data to indices
max_len = 50
X = tokens_to_indices(df_clean['tokens'].tolist(), vocab, max_len)
y = df_clean['score_shifted'].values

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

### 1.6. Train/test split

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Score distribution (train): {Counter(y_train)}')

## 2. Prepare GloVe embeddings

GloVe (Global Vectors) provides pretrained word embeddings trained on large text corpora. Using these gives our model a head start - words already have meaningful representations instead of random vectors.

### 2.1. Load embeddings into dictionary

In [ ]:
glove_vectors = {}

with open(glove_file, 'r', encoding='utf-8') as f:
    for line in f:

        values = line.split()
        word = values[0]
        vector = np.array(values[1:], dtype='float32')
        glove_vectors[word] = vector

embedding_dim = len(next(iter(glove_vectors.values())))

print(f'Loaded {len(glove_vectors):,} word vectors')
print(f'Embedding dimension: {embedding_dim}')

### 2.2. Create embedding look-up table for our vocabulary

In [ ]:
vocab_size = len(vocab)

embedding_matrix = np.zeros((vocab_size, embedding_dim))
words_found = 0

for word, idx in vocab.items():
    if word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
        words_found += 1

    else:
        # Random initialization for words not in GloVe
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

coverage = words_found / vocab_size
print(f'Vocabulary coverage: {words_found:,} / {vocab_size:,} ({coverage:.1%})')
print(f'Embedding matrix shape: {embedding_matrix.shape}')

## 3. Baseline: Bidirectional GRU model

First, let's establish our baseline performance with a pure recurrent architecture.

### 3.1. Build model

In [ ]:
# Convert numpy embedding matrix into a Constant initializer
glove_initializer = keras.initializers.Constant(embedding_matrix)

# Build Bidirectional GRU model with pretrained GloVe embeddings
baseline_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(
        input_dim=vocab_size, 
        output_dim=embedding_dim,
        embeddings_initializer=glove_initializer,
        trainable=True  # Set to True to make embeddings trainable
    ),
    SpatialDropout1D(dropout),
    Bidirectional(GRU(hidden_dim, return_sequences=True, name='GRU_1'), name='Bidirectional_1'),
    Bidirectional(GRU(hidden_dim, return_sequences=False, name='GRU_2'), name='Bidirectional_2'),
    Dropout(dropout),
    Dense(len(set(y_train)), activation='softmax', kernel_regularizer=keras.regularizers.l2(l2_reg))
])

baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

baseline_params = baseline_model.count_params()
baseline_model.summary()

### 3.2. Define training callbacks

In [ ]:
callbacks = [
    ModelCheckpoint(
        '../models/activity_rnn_classifier.keras',
        save_best_only=True,
        monitor='val_accuracy',
        mode='max',
        verbose=verbose
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=early_stopping_patience,
        restore_best_weights=True,
        verbose=verbose
    ),
    TensorBoard(
        log_dir='../logs/activity_rnn_classifier',
        histogram_freq=1,
        write_graph=True
    )
]

### 3.3. Calculate class weights

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f'Class weights: {class_weight_dict}')

### 3.4. Train the model

In [20]:
%%time

# Training with class weighting and early stopping
history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=epochs,
    batch_size=batch_size,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=verbose
)

print()

KeyboardInterrupt: 

### 3.5. Baseline model learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Baseline evaluation

Record the baseline metrics below so you can compare them to your CNN-RNN model later.

### 4.1. Make test set predictions

In [ ]:
# Load best model and get predictions
baseline_model = keras.saving.load_model('../models/rnn_classifier.keras')

# Get predicted probabilities and classes
y_prob_baseline = baseline_model.predict(X_test)
y_pred_baseline = y_prob_baseline.argmax(axis=1)

# Class labels
class_names = ['-2', '-1', '0', '1', '2']
n_classes = len(class_names)

print(f'Test samples: {len(y_test)}')
print(f'Prediction distribution:\n')
print(f'  Actual:    {Counter(y_test)}')
print(f'  Predicted: {Counter(y_pred_baseline)}')

### 4.2. Per-class accuracy

In [ ]:
# Per-class accuracy table
per_class_stats = []

for i, class_name in enumerate(class_names):
    mask = y_test == i
    n_samples = mask.sum()
    n_correct = (y_pred_baseline[mask] == i).sum()
    acc = n_correct / n_samples if n_samples > 0 else 0
    
    per_class_stats.append({
        'Class': class_name,
        'Samples': n_samples,
        'Correct': n_correct,
        'Accuracy': f'{acc:.1%}'
    })

# Add overall accuracy
baseline_acc = (y_pred_baseline == y_test).mean()

per_class_stats.append({
    'Class': 'Overall',
    'Samples': len(y_test),
    'Correct': (y_pred_baseline == y_test).sum(),
    'Accuracy': f'{baseline_acc:.1%}'
})

accuracy_df = pd.DataFrame(per_class_stats)
accuracy_df

### 4.3. Confusion matrix

In [ ]:
# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))

ax.set_title('Confusion matrix')
cm = confusion_matrix(y_test, y_pred_baseline)

sns.heatmap(
    cm, 
    annot=True, fmt='d', cmap='Blues', 
    xticklabels=class_names, yticklabels=class_names, ax=ax
)

ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

### 4.4. Predicted probability distributions by class

In [ ]:
# Predicted probability distributions for each true class
fig, axes = plt.subplots(1, n_classes, figsize=(10, 2.5))

plt.suptitle('Predicted probability distributions by true class', y=1.02)

for i, class_name in enumerate(class_names):

    # Get predictions for samples of this true class
    mask = y_test == i
    probs_for_class = y_prob_baseline[mask]
    
    # Plot distribution of predicted probabilities for each predicted class
    ax = axes[i]
    ax.boxplot([probs_for_class[:, j] for j in range(n_classes)], tick_labels=class_names)
    ax.set_title(f'True class: {class_name}')
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

### 4.5. Evaluation curves

In [ ]:
# Binarize labels for multiclass ROC/PR
y_test_bin = label_binarize(y_test, classes=range(n_classes))

# Compute ROC and PR curves for each class
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# ROC curves
ax_roc = axes[0]
ax_roc.set_title('ROC curves (one-vs-rest)')

for i, class_name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_baseline[:, i])
    roc_auc = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, label=f'Class {class_name} (AUC={roc_auc:.2f})')

ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax_roc.set_xlabel('False positive rate')
ax_roc.set_ylabel('True positive rate')
ax_roc.legend(loc='lower right')
ax_roc.set_xlim([0, 1])
ax_roc.set_ylim([0, 1.05])

# PR curves
ax_pr = axes[1]
ax_pr.set_title('Precision-recall curves (one-vs-rest)')

for i, class_name in enumerate(class_names):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_prob_baseline[:, i])
    pr_auc = auc(recall, precision)
    ax_pr.plot(recall, precision, label=f'Class {class_name} (AUC={pr_auc:.2f})')

ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_xlim([0, 1])
ax_pr.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

---

## 5. Activity: Build a CNN-RNN hybrid model (SOLUTION)

The solution below demonstrates one effective approach: adding a Conv1D layer before the GRU to capture local n-gram patterns.

### Why add convolutions?

- **Local pattern detection**: Conv1D layers can detect n-gram patterns (e.g., "not good", "very happy") that are important for sentiment
- **Dimensionality reduction**: Convolutions can reduce sequence length before the RNN, making training faster
- **Feature extraction**: The CNN extracts local features, then the RNN captures long-range dependencies

### Solution architecture

```
Embedding (vocab_size × 100)
    ↓
SpatialDropout1D (0.3)
    ↓
Conv1D (128 filters, kernel_size=3, relu, padding='same')  ← Captures trigram patterns
    ↓
MaxPooling1D (pool_size=2)  ← Reduces sequence length by half
    ↓
Bidirectional GRU (32 units, return_sequences=True)
    ↓
Bidirectional GRU (16 units)
    ↓
Dropout (0.3)
    ↓
Dense (5 classes, softmax)
```

### Key design decisions

1. **128 filters**: Each filter learns a different pattern detector. More filters = more patterns, but more parameters.
2. **kernel_size=3**: Captures trigrams like "not very good". You could try 4 or 5 for longer patterns.
3. **padding='same'**: Output length = input length, so MaxPooling works correctly.
4. **MaxPooling1D(pool_size=2)**: Halves the sequence length, making the GRU faster and helping it focus on the most salient features.

### 5.1. Define the CNN-RNN model

In [ ]:
from keras.layers import Conv1D, MaxPooling1D

# Model hyperparameters
conv_filters     = 128
conv_kernel_size = 3
max_pool_size    = 2

# Convert numpy embedding matrix into a Constant initializer
glove_initializer = keras.initializers.Constant(embedding_matrix)

# Build Bidirectional GRU model with pretrained GloVe embeddings
cnn_rnn_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(
        input_dim=vocab_size, 
        output_dim=embedding_dim,
        embeddings_initializer=glove_initializer,
        trainable=True  # Set to True to make embeddings trainable
    ),
    SpatialDropout1D(dropout),
    Conv1D(filters=conv_filters, kernel_size=conv_kernel_size, activation='relu', padding='same'),
    MaxPooling1D(pool_size=max_pool_size),
    Bidirectional(GRU(hidden_dim, return_sequences=True, name='GRU_1'), name='Bidirectional_1'),
    Bidirectional(GRU(hidden_dim, return_sequences=False, name='GRU_2'), name='Bidirectional_2'),
    Dropout(dropout),
    Dense(len(set(y_train)), activation='softmax', kernel_regularizer=keras.regularizers.l2(l2_reg))
])

cnn_rnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_rnn_params = cnn_rnn_model.count_params()
cnn_rnn_model.summary()

### 5.2. Define training callbacks

In [ ]:
# Define callbacks for CNN-RNN model (saves to different file)
cnn_rnn_callbacks = [
    ModelCheckpoint(
        '../models/cnn_rnn_classifier.keras',
        save_best_only=True,
        monitor='val_accuracy',
        mode='max',
        verbose=verbose
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=early_stopping_patience,
        restore_best_weights=True,
        verbose=verbose
    ),
    TensorBoard(
        log_dir='../logs/cnn-rnn_classifier',
        histogram_freq=1,
        write_graph=True
    )
]

### 5.3. Train your CNN-RNN model

In [ ]:
%%time

# Train the CNN-RNN model
cnn_rnn_history = cnn_rnn_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=epochs,
    batch_size=batch_size,
    class_weight=class_weight_dict,
    callbacks=cnn_rnn_callbacks,
    verbose=verbose
)

print()

### 5.4. CNN-RNN model learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.5. Evaluate your CNN-RNN model

In [ ]:
# Load best CNN-RNN model and get predictions
cnn_rnn_model = keras.saving.load_model('../models/cnn_rnn_classifier.keras')

# Get predicted probabilities and classes
y_prob_cnn_rnn = cnn_rnn_model.predict(X_test)
y_pred_cnn_rnn = y_prob_cnn_rnn.argmax(axis=1)

# Calculate accuracy
cnn_rnn_acc = (y_pred_cnn_rnn == y_test).mean()
print(f'CNN-RNN model accuracy: {cnn_rnn_acc:.1%}')

---

## 6. Compare models

Now let's compare your CNN-RNN model to the baseline GRU model.

In [ ]:
# Create comparison dataframe
comparison = pd.DataFrame({
    'Model': ['Baseline GRU', 'CNN-RNN'],
    'Accuracy': [f'{baseline_acc:.1%}', f'{cnn_rnn_acc:.1%}'],
    'Parameters': [baseline_params, cnn_rnn_params]
})

print('Model comparison:')
comparison

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(
    axes, 
    [y_pred_baseline, y_pred_cnn_rnn], 
    ['Baseline GRU', 'CNN-RNN']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names, ax=ax
    )
    ax.set_title(f'{title} (Acc: {(y_pred == y_test).mean():.1%})')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 7. Reflection

**1. Did your CNN-RNN model outperform the baseline?**

Results vary by run, but typically the CNN-RNN model shows marginal improvement or similar performance. The convolutional layer helps by:
- Detecting local sentiment indicators like "not good" or "very happy" as fixed patterns
- Reducing sequence length before the RNN, which can help with longer sequences
- Extracting position-invariant features (the same pattern is detected anywhere in the tweet)

However, for short tweets (average ~10-15 tokens), the baseline GRU is already quite capable of capturing the full context, limiting the CNN's advantage.

**2. How did the number of parameters change?**

The CNN-RNN model typically has **more parameters** due to the Conv1D layer (128 filters × 3 kernel × 100 embedding_dim = 38,400 weights + biases). However, we reduced the GRU sizes to partially compensate.

Trade-off: More parameters = more capacity to learn complex patterns, but also higher risk of overfitting on small datasets. With ~15K training samples, regularization (dropout, L2) is important.

**3. Which classes improved the most?**

Typically, the extreme sentiment classes (-2 and +2) and the neutral class (0) see the most variability:
- **Extreme classes**: Often underrepresented, so small changes in the model can have outsized effects
- **Neutral class**: Hard to distinguish from mild sentiment; n-gram patterns like "it's okay" or "not bad" can help

The adjacent classes (-1, +1) are hardest to improve because they're semantically closest to neutral.

**4. What other architectures could you try?**

- **Multiple kernel sizes**: Use parallel Conv1D layers with kernel_size=2,3,4,5 to capture different n-gram lengths, then concatenate (requires Functional API)
- **Deeper convolutions**: Stack multiple Conv1D layers with increasing filters (64 → 128 → 256)
- **Pure CNN**: Replace GRU entirely with GlobalMaxPooling1D; often competitive and much faster to train
- **Attention mechanisms**: Add self-attention after the RNN to let the model focus on important words
- **Transformer-based**: Use a pretrained model like BERT or DistilBERT for state-of-the-art results